# Vidal Case 2 + Active Learning: minimal confirmation
Runs full data, 10% random, and 10% Regression-CLUE for one transfer pair.

In [ ]:
from google.colab import drive, userdata
from pathlib import Path
import subprocess, sys

drive.mount('/content/drive')

ACTIVE_REPO_URL = 'https://github.com/zhangcastle/battery-soc-active-learning.git'
REPO = Path('/content/battery-soc-active-learning')
github_token = None
for secret_name in ('GITHUB_TOKEN', 'GITHUB_1'):
    try:
        github_token = userdata.get(secret_name)
    except Exception:
        continue
    if github_token:
        break
if not github_token:
    raise RuntimeError('Enable a Colab secret named GITHUB_TOKEN or GITHUB_1.')
authenticated_url = ACTIVE_REPO_URL.replace('https://', f'https://x-access-token:{github_token}@')
try:
    if (REPO / '.git').is_dir():
        subprocess.run(['git', 'remote', 'set-url', 'origin', authenticated_url], cwd=REPO, check=True)
        subprocess.run(['git', 'pull', '--ff-only'], cwd=REPO, check=True)
    else:
        subprocess.run(['git', 'clone', '--depth', '1', authenticated_url, str(REPO)], check=True)
except subprocess.CalledProcessError:
    raise RuntimeError('GitHub clone/pull failed. Check the GITHUB_TOKEN repository permission.') from None
finally:
    if (REPO / '.git').is_dir():
        subprocess.run(['git', 'remote', 'set-url', 'origin', ACTIVE_REPO_URL], cwd=REPO, check=True)
    del authenticated_url, github_token
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)

In [ ]:
DRIVE = Path('/content/drive/MyDrive/battery_soc_benchmark')
INPUT = DRIVE / 'data/labelled/vidal_sequence_v2'
CASE1 = DRIVE / 'outputs/vidal_case1'
OUTPUT = DRIVE / 'outputs/active_learning_confirmation'
SOURCE, TARGET = 'lg_hg2', 'samsung_30t'

def latest_checkpoint(seed):
    matches = sorted((CASE1 / SOURCE / f'seed_{seed}').glob('*/checkpoints/epoch_515.pt'))
    return matches[-1] if matches else None

SOURCE_CHECKPOINT = latest_checkpoint(42)
if SOURCE_CHECKPOINT is None:
    raise FileNotFoundError(f'Missing existing formal Case 1 checkpoint under {CASE1 / SOURCE / "seed_42"}')
print('Existing source checkpoint:', SOURCE_CHECKPOINT)

In [ ]:
import subprocess, sys

COMMITTEE_CHECKPOINT = latest_checkpoint(43)
if COMMITTEE_CHECKPOINT is None:
    print('No seed_43 source checkpoint; training it once now...')
    subprocess.run([
        sys.executable, str(REPO / 'scripts/reproductions/vidal_2019/run_case1.py'),
        '--dataset', SOURCE,
        '--config', str(REPO / 'configs/reproductions/vidal_2019/vidal_case1_lg_hg2.yaml'),
        '--input-root', str(INPUT), '--output-root', str(CASE1),
        '--run-mode', 'formal', '--seed', '43', '--device', 'auto',
    ], cwd=REPO, check=True)
    COMMITTEE_CHECKPOINT = latest_checkpoint(43)
if COMMITTEE_CHECKPOINT is None:
    raise RuntimeError('Seed 43 training finished without epoch_515.pt')
print('Committee checkpoint:', COMMITTEE_CHECKPOINT)

In [ ]:
subprocess.run([
    sys.executable, str(REPO / 'scripts/run_active_learning_confirmation.py'),
    '--source', SOURCE, '--target', TARGET,
    '--config', str(REPO / 'configs/reproductions/vidal_2019/vidal_case2_diagnostic.yaml'),
    '--input-root', str(INPUT), '--output-root', str(OUTPUT),
    '--source-checkpoint', str(SOURCE_CHECKPOINT),
    '--committee-checkpoint', str(COMMITTEE_CHECKPOINT),
    '--budget', '0.10', '--run-mode', 'smoke', '--device', 'auto',
], cwd=REPO, check=True)

In [ ]:
import pandas as pd
pd.read_csv(OUTPUT / 'confirmation_summary.csv')